<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/03_Advanced/L37_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L37: Data Preparation - Cleaning and Dataset Creation

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Advanced  
**Lesson:** 37 of 45

---

## Learning Objectives

By the end of this lesson, you will:
- Clean and filter raw text (length, language, quality)
- Deduplicate documents or spans
- Build a training dataset from raw text with HuggingFace datasets

---

## Concept: Data Preparation

**Cleaning**: remove empty lines, normalize whitespace, filter by length. **Filtering**: language detection, toxicity/quality filters. **Deduplication**: exact or near-duplicate removal (hashing or embeddings). We use datasets and simple Python to illustrate.

---

In [ ]:
!pip install datasets -q
from datasets import load_dataset, Dataset
import re
import hashlib
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Load and Clean Text

---

In [ ]:
def clean_text(text):
    if not text or not text.strip():
        return None
    text = re.sub(r"\s+", " ", text.strip())
    return text if len(text) >= 20 else None

ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train", trust_remote_code=True)
ds = ds.filter(lambda x: clean_text(x["text"]) is not None)
ds = ds.map(lambda x: {"text": clean_text(x["text"])}, remove_columns=[c for c in ds.column_names if c != "text"])
print(f"After cleaning: {len(ds)} lines")

## Part 2: Deduplication by Hash

---

In [ ]:
def dedupe_by_hash(ds, text_key="text"):
    seen = set()
    indices = []
    for i, row in enumerate(ds):
        h = hashlib.sha256(row[text_key].encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            indices.append(i)
    return ds.select(indices)

ds_deduped = dedupe_by_hash(ds)
print(f"After dedup: {len(ds_deduped)} (was {len(ds)})")

## Part 3: Filter by Length and Create Final Dataset

---

In [ ]:
ds_final = ds_deduped.filter(lambda x: 50 <= len(x["text"].split()) <= 500)
print(f"Length filter (50–500 words): {len(ds_final)} examples")
print("Sample:", ds_final[0]["text"][:100] + "...")

## Exercises

1. Add a simple language filter (e.g. langdetect) and keep one language.
2. Use minhash or embeddings for near-duplicate detection.
3. Build a single "document" by concatenating chunks with EOS and save as one text file for LM training.

---

## Key Takeaways

1. Clean (whitespace, empty, min length) before training.
2. Dedupe to avoid memorization and reduce dataset size.
3. Length and quality filters improve data utility.

---

## Next Lesson

**L38: Tokenizer Training** – Training BPE/WordPiece on custom corpora.

---